# 04 Baselines and Logistic Regression

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
RESULTS = Path("../results")
spec = json.loads((MODELS / "features.json").read_text())
FEATURES, CATEGORICAL = spec["features"], spec["categorical"]

X_train = pd.read_parquet(PROCESSED / "X_train.parquet")[FEATURES]
X_val = pd.read_parquet(PROCESSED / "X_val.parquet")[FEATURES]
y_train = pd.read_parquet(PROCESSED / "y_train.parquet").is_churn.values
y_val = pd.read_parquet(PROCESSED / "y_val.parquet").is_churn.values
print(X_train.shape, X_val.shape)

val_rows = []
def record(name, p):
    row = {"model": name, "pr_auc": average_precision_score(y_val, p), "roc_auc": roc_auc_score(y_val, p),
           "log_loss": log_loss(y_val, np.clip(p, 1e-6, 1 - 1e-6)), "f1@0.5": f1_score(y_val, p >= 0.5)}
    val_rows.append(row)
    print({k: (round(v, 4) if k != "model" else v) for k, v in row.items()})

(662679, 33) (94669, 33)


## Majority Baseline

In [2]:
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
record("Majority (prior)", dummy.predict_proba(X_val)[:, 1])

{'model': 'Majority (prior)', 'pr_auc': 0.0899, 'roc_auc': 0.5, 'log_loss': 0.3024, 'f1@0.5': 0.0}


## Rule Baseline

In [3]:
# churn probability = training churn rate of the subscriber's auto-renew group
rule_rates = pd.Series(y_train).groupby(X_train.last_auto_renew.fillna(0).values).mean()
print(rule_rates)
record("Rule: auto-renew off", X_val.last_auto_renew.fillna(0).map(rule_rates).values)

0.0    0.457131
1.0    0.038019
dtype: float64
{'model': 'Rule: auto-renew off', 'pr_auc': 0.3162, 'roc_auc': 0.7744, 'log_loss': 0.2287, 'f1@0.5': 0.0}


## Logistic Regression Pipeline

In [4]:
# every learned step sits inside the pipeline, so it is fitted on the training split only
SKEWED = ["tx_count", "total_paid", "cancel_count", "discount_count", "last_amount_paid", "total_hours", "secs_per_day", "songs_per_day"]
OTHER_NUM = [c for c in FEATURES if c not in SKEWED + CATEGORICAL]
linear_prep = ColumnTransformer([
    ("skewed", make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()), SKEWED),
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), OTHER_NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", min_frequency=100)), CATEGORICAL),
])

## Logistic Regression Tuning

In [5]:
tuning, best_ll = [], np.inf
for C in [0.01, 0.1, 1.0, 10.0]:
    m = make_pipeline(linear_prep, LogisticRegression(C=C, max_iter=2000)).fit(X_train, y_train)
    ll = log_loss(y_val, m.predict_proba(X_val)[:, 1])
    tuning.append({"C": C, "val_log_loss": ll})
    print(f"C={C:<5} validation log loss {ll:.5f}")
    if ll < best_ll:
        best_ll, logreg = ll, m
record("Logistic regression", logreg.predict_proba(X_val)[:, 1])

coef = pd.Series(logreg[-1].coef_[0], index=logreg[0].get_feature_names_out())
coef.sort_values(key=abs, ascending=False).head(10).round(3)

C=0.01  validation log loss 0.08810


C=0.1   validation log loss 0.08669


C=1.0   validation log loss 0.08614


C=10.0  validation log loss 0.08606
{'model': 'Logistic regression', 'pr_auc': 0.8991, 'roc_auc': 0.976, 'log_loss': 0.0854, 'f1@0.5': 0.8512}


num__days_until_expiration                     4.373
cat__last_payment_method_17.0                  4.251
num__last_plan_days                           -3.679
cat__last_payment_method_19.0                 -3.522
num__days_since_last_tx                        3.315
num__avg_plan_days                             3.310
cat__last_payment_method_32.0                  3.286
cat__last_payment_method_infrequent_sklearn   -3.050
cat__last_payment_method_15.0                  2.335
cat__last_payment_method_35.0                 -2.254
dtype: float64

## Save

In [6]:
joblib.dump({"dummy": dummy, "rule_rates": rule_rates.to_dict(), "logreg": logreg}, MODELS / "baselines.joblib")
pd.DataFrame(val_rows).to_csv(RESULTS / "validation_baselines.csv", index=False)
(RESULTS / "tuning_logreg.json").write_text(json.dumps(tuning))
pd.DataFrame(val_rows).set_index("model").round(4)

,pr_auc,roc_auc,log_loss,f1@0.5
model,,,,
Majority (prior),0.0899,0.5000,0.3024,0.0000
Rule: auto-renew off,0.3162,0.7744,0.2287,0.0000
Logistic regression,0.8991,0.9760,0.0854,0.8512
